# Seoul Station Stacking V20 No-Peak

Seoul (`RKSI`) adaptation of the KDAL V20-aligned station-stacking workflow. The notebook uses the existing Asia 11 AM parquet contract, Fahrenheit-native modeling, Wunderground-only settlement highs, and an expanding 2022–2025 validation design.


In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "asia_station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/asia_station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CITY_ID = "seoul"
CITY_LABEL = "Seoul"
STATION_ID = "RKSI"
TIMEZONE = "Asia/Seoul"
DATA_ROOT = PROJECT_ROOT / "data" / "calibration" / "asia_11am"
OUTPUT_DIR = DATA_ROOT / "models" / f"v20_{CITY_ID}_no_peak"
PROVIDERS = ("gfs", "gefs", "jma_msm")
TIMING_MODE = "asia_same_day_11am_live_safe"
FEATURE_VERSION = "v20_asia_no_peak"
TRAINING_PROFILE = "v20_aligned"
TARGET_SOURCE = "wunderground_only"
TARGET_MODE = "remaining_warmup"
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
FAST_MODE = False
EXPORT_MODEL_WEIGHTS = True
MODEL_VERSION = f"station_high_regressor_v20_{CITY_ID}_no_peak_stack"
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.calibration.asia_station_stacking import (
    ASIA_PROVIDERS,
    ASIA_TEST_YEAR,
    ASIA_TIMING_MODE,
    asia_expanding_folds,
    build_asia_station_wide_dataset,
    provider_readiness,
)
from src.calibration.station_stacking import (
    StationStackingConfig,
    V20_ASIA_NO_PEAK_FEATURE_VERSION,
    missing_model_dependencies,
    run_station_year_split_experiment,
)
from src.export_station_stacking_v2_models import export_station_model_weights


## City contract

- Existing Asia parquet data rooted at `data/calibration/asia_11am`
- Local 11 AM live-safe observation cutoff
- GFS, GEFS, and JMA MSM forecast inputs
- Wunderground-only daily settlement high target
- Fahrenheit-native model values with Celsius reporting


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
            "validation_weight": 1.0,
        }
        for fold in asia_expanding_folds()
    ]
)
fold_spec


,fold,train_start_year,train_end_year,validation_year,validation_weight
0,fold_2022_to_2023,2022,2022,2023,1.0
1,fold_2022_2023_to_2024,2022,2023,2024,1.0
2,fold_2022_2024_to_2025,2022,2024,2025,1.0


## Provider readiness


In [4]:
readiness = provider_readiness(DATA_ROOT, CITY_ID, providers=PROVIDERS)
readiness


,city_id,provider,row_count,ok_count,first_contract_date,last_contract_date,ready
0,seoul,gfs,19318,19318,2022-07-03,2026-07-27,True
1,seoul,gefs,1484,1484,2022-07-03,2026-07-25,True
2,seoul,jma_msm,19318,19318,2022-07-03,2026-07-27,True


## Build the live-safe modeling frame


In [5]:
features = build_asia_station_wide_dataset(
    DATA_ROOT,
    CITY_ID,
    feature_version=FEATURE_VERSION,
    providers=PROVIDERS,
)
features[[
    "contract_date",
    "actual_high_f",
    "observed_high_temp_through_as_of_f",
    "gfs_high_f",
    "gefs_high_f",
    "jma_msm_high_f",
    "strict_quality_ok",
]].head()


,contract_date,actual_high_f,observed_high_temp_through_as_of_f,gfs_high_f,gefs_high_f,jma_msm_high_f,strict_quality_ok
0,2022-07-03,86.0,84.2,74.932283,80.602188,75.56,True
1,2022-07-04,87.8,87.8,74.304499,79.524538,77.00,True
2,2022-07-05,87.8,86.0,74.177546,79.903892,77.90,True
3,2022-07-06,91.4,86.0,79.379476,82.127001,80.06,True
4,2022-07-07,86.0,84.2,76.332028,79.228204,79.88,True


In [6]:
feature_coverage = (
    features[["actual_high_f", "observed_high_temp_through_as_of_f", *[f"{p}_high_f" for p in PROVIDERS]]]
    .notna()
    .mean()
    .rename("non_null_fraction")
    .to_frame()
)
feature_coverage


,non_null_fraction
actual_high_f,1.000000
observed_high_temp_through_as_of_f,0.998654
gfs_high_f,1.000000
gefs_high_f,0.998654
jma_msm_high_f,1.000000


## Train and score


In [7]:
missing_packages = missing_model_dependencies(("xgboost", "lightgbm", "catboost", "optuna"))
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric="mae_f",
    feature_version=FEATURE_VERSION,
    training_profile=TRAINING_PROFILE,
    target_mode=TARGET_MODE,
    target_source=TARGET_SOURCE,
    max_feature_missing_fraction=0.03,
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=asia_expanding_folds(),
    year_split_validation_weights={2023: 1.0, 2024: 1.0, 2025: 1.0},
    year_split_test_train_years=(2022, 2025),
    year_split_test_year=ASIA_TEST_YEAR,
    output_dir=OUTPUT_DIR,
    prebuilt_features=features,
)
config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/asia_11am/models/v20_seoul_no_peak/RKSI_optuna.sqlite3')

In [8]:
result = run_station_year_split_experiment(config)
result.scoreboard


d:\dev\weather-research\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\dev\weather-research\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\dev\weather-research\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
d:\dev\weather-research\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X

,period,method,count,mae_f,rmse_f
0,validation_2023_2025,xgboost,1094,1.331729,1.756886
1,validation_2023_2025,lightgbm,1094,1.328496,1.767506
2,validation_2023_2025,catboost,1094,1.343982,1.789345
3,validation_2023_2025,provider_mean,1094,5.608588,6.851075
4,validation_2023_2025,provider_median,1094,5.662443,6.950712
5,validation_2023_2025,gfs_raw,1094,6.639161,8.055828
6,test_2026,xgboost,203,1.245299,1.637463
7,test_2026,lightgbm,203,1.340487,1.734650
8,test_2026,catboost,203,1.387389,1.773165
9,test_2026,ridge_stack,203,1.290192,1.667873


## Celsius reporting and export


In [9]:
celsius_predictions = result.test_predictions.copy()
for column in ("actual_high_f", "predicted_high_f", "error_f"):
    if column in celsius_predictions:
        celsius_predictions[column.replace("_f", "_c")] = pd.to_numeric(celsius_predictions[column], errors="coerce") * 5.0 / 9.0
celsius_predictions.head()


,contract_date,fold,method,param_key,evaluation_scope,actual_high_f,predicted_high_f,error_f,absolute_error_f,actual_high_c,predicted_high_c,error_c
0,2026-01-01,train_2022_2025_test_2026,gfs_raw,<NA>,year_split_test,24.8,27.159408,-2.359408,2.359408,13.777778,15.088560,-1.310782
1,2026-01-02,train_2022_2025_test_2026,gfs_raw,<NA>,year_split_test,24.8,26.971389,-2.171389,2.171389,13.777778,14.984105,-1.206327
2,2026-01-03,train_2022_2025_test_2026,gfs_raw,<NA>,year_split_test,32.0,36.900883,-4.900883,4.900883,17.777778,20.500491,-2.722713
3,2026-01-04,train_2022_2025_test_2026,gfs_raw,<NA>,year_split_test,37.4,36.748300,0.651700,0.651700,20.777778,20.415722,0.362056
4,2026-01-05,train_2022_2025_test_2026,gfs_raw,<NA>,year_split_test,33.8,32.982822,0.817178,0.817178,18.777778,18.323790,0.453988


In [10]:
if EXPORT_MODEL_WEIGHTS:
    exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        city_id=CITY_ID,
        artifact_dir=config.resolved_output_dir(),
        model_version=MODEL_VERSION,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        training_profile=config.effective_training_profile,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        source_pipeline=f"notebooks/experiments/station_stacking_v20_asia_no_peak/{CITY_ID}",
    )
    exported_weights.bundle_path, exported_weights.manifest_path
else:
    print("Model export disabled for this notebook.")


In [11]:
result.output_paths


{'features': WindowsPath('D:/dev/weather-research/data/calibration/asia_11am/models/v20_seoul_no_peak/RKSI_features.csv'),
 'year_split_tuning': WindowsPath('D:/dev/weather-research/data/calibration/asia_11am/models/v20_seoul_no_peak/RKSI_year_split_tuning.csv'),
 'year_split_validation_predictions': WindowsPath('D:/dev/weather-research/data/calibration/asia_11am/models/v20_seoul_no_peak/RKSI_year_split_validation_predictions.csv'),
 'year_split_test_predictions': WindowsPath('D:/dev/weather-research/data/calibration/asia_11am/models/v20_seoul_no_peak/RKSI_year_split_test_predictions.csv'),
 'year_split_metrics': WindowsPath('D:/dev/weather-research/data/calibration/asia_11am/models/v20_seoul_no_peak/RKSI_year_split_metrics.csv'),
 'year_split_selected_hyperparameters': WindowsPath('D:/dev/weather-research/data/calibration/asia_11am/models/v20_seoul_no_peak/RKSI_year_split_selected_hyperparameters.csv'),
 'year_split_feature_importance': WindowsPath('D:/dev/weather-research/data/calibr